<a href="https://colab.research.google.com/github/routparam12/Python_qns/blob/main/Differentname.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.6 MB/s eta 0:00:00


In [2]:
import pandas as pd

# Read CSV
df = pd.read_csv("/content/panelists.csv")

# Normalize text (lowercase + handle missing values)
df["firstName"] = df["firstName"].fillna("").str.lower().str.strip()
df["lastName"] = df["lastName"].fillna("").str.lower().str.strip()
df["email"] = df["email"].fillna("").str.lower().str.strip()

# Extract email username (before @)
df["email_name"] = df["email"].str.split("@").str[0]

# Find emails where email username doesn't contain firstname or lastname
mask = ~(
    df.apply(
        lambda row: (
            row["firstName"] in row["email_name"]
            or row["lastName"] in row["email_name"]
        ),
        axis=1
    )
)

# Get suspicious emails
result = df.loc[mask, ["firstName", "lastName", "email"]]

print(result)

/tmp/ipykernel_4749/2502985963.py:4: DtypeWarning: Columns (20,23,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/panelists.csv")


       firstName    lastName                               email
0          jakub   olszewski  1748613780618_gamekrzywy@gmail.com
1         ronald      martin                heskamran4@gmail.com
4         nathan       lloyd              ngamemasterl@gmail.com
8           alex     martens                 1234zamiz@gmail.com
9       jeanette  sutherland                    jmms47@gmail.com
...          ...         ...                                 ...
123709    joseph      irikwu           dosantomario382@gmail.com
123710     davis       walis         sahorinaoiche1011@gmail.com
123711  eileen f      ashton     johnathanjohnathan735@gmail.com
123716     jason      vargas        tanvirahmedsujon42@gmail.com
123719     tanya      miller            crabtreesally0@gmail.com

[36816 rows x 3 columns]


In [5]:
import pandas as pd

# Read CSV
df = pd.read_csv("/content/panelists.csv")

# Normalize columns
df["firstName"] = df["firstName"].fillna("").str.lower().str.strip()
df["lastName"] = df["lastName"].fillna("").str.lower().str.strip()
df["email"] = df["email"].fillna("").str.lower().str.strip()

# Extract email username (before @)
df["email_name"] = df["email"].str.split("@").str[0]

# Email mismatch condition
email_mask = ~df.apply(
    lambda row: (
        row["firstName"] in row["email_name"]
        or row["lastName"] in row["email_name"]
    ),
    axis=1
)

# Country filter
country_mask = df["countryId"] != 3

# Combine filters
filtered_df = df.loc[
    email_mask & country_mask,
    ["panelistId", "firstName", "lastName", "email"]
]


# Save to CSV
output_file = "filtered_emails2.csv"
filtered_df.to_csv(output_file, index=False)

print(f"Saved {len(filtered_df)} rows to {output_file}")

/tmp/ipykernel_4749/158194380.py:4: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/panelists.csv")


Saved 46213 rows to filtered_emails.csv


In [8]:
import pandas as pd

# Read CSV
df = pd.read_csv("/content/panelists.csv")

# Handle nulls + normalize case
df["email"] = df["email"].fillna("").str.lower()
df["firstName"] = df["firstName"].fillna("").str.lower()
df["lastName"] = df["lastName"].fillna("").str.lower()

# Apply filters
filtered_df = df[
    (df["email"].str.contains("hosen", na=False)) &
    (~df["lastName"].str.contains("hosen", na=False)) &
    (~df["firstName"].str.contains("hosen", na=False)) &
    (df["countryId"] != 3)
][["panelistId", "firstName", "lastName", "email"]]

# Save output
output_file = "filtered_hosen.csv"
filtered_df.to_csv(output_file, index=False)

print(f"Saved {len(filtered_df)} rows to {output_file}")

/tmp/ipykernel_4749/3527784732.py:4: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/panelists.csv")


Saved 21 rows to filtered_hosen.csv


In [2]:
import pandas as pd
import re
from rapidfuzz import fuzz


def should_remove(firstName, lastName, email):
    username = email.split("@")[0].lower()

    # split names into words
    names = re.findall(r"[a-z]+", f"{firstName} {lastName}".lower())

    for name in names:
        if len(name) < 3:
            continue

        # exact match
        if name in username:
            return True

        # fuzzy match
        if fuzz.partial_ratio(name, username) >= 85:
            return True

    return False


# Load CSV
df = pd.read_excel("/content/sus2.xlsx")

# Keep only rows that DON'T match
filtered_df = df[
    ~df.apply(
        lambda x: should_remove(
            x["firstName"],
            x["lastName"],
            x["email"]
        ),
        axis=1
    )
]

filtered_df.to_csv("output.csv", index=False)

print("Done!")

Done!


In [2]:
pip install rapidfuzz Levenshtein jellyfish

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 18.1 MB/s eta 0:00:00


In [12]:
import pandas as pd
import re
from rapidfuzz import fuzz
import jellyfish


class EmailMatcher:

    def __init__(
        self,
        csv_path,
        email_col="email",
        panelist_col="panelistId"
    ):
        self.email_col = email_col
        self.panelist_col = panelist_col

        self.df = pd.read_csv(csv_path)

        self.df["username"] = (
            self.df[email_col]
            .fillna("")
            .astype(str)
            .apply(self._clean_email)
        )

    # -------------------------
    # Normalize
    # -------------------------
    def _clean_email(self, email):
        username = str(email).lower().split("@")[0]

        # keep only letters
        username = re.sub(
            r"[^a-z]",
            "",
            username
        )

        return username

    # -------------------------
    # Score Logic
    # -------------------------
    def _score(
        self,
        query,
        candidate
    ):

        if not query or not candidate:
            return 0

        # 1. exact similarity
        ratio_score = fuzz.ratio(
            query,
            candidate
        )

        # 2. typo tolerant
        partial_score = fuzz.partial_ratio(
            query,
            candidate
        )

        # 3. prefix bonus
        prefix_bonus = 0

        if (
            query[:5]
            ==
            candidate[:5]
        ):
            prefix_bonus += 15

        # 4. substring bonus
        substring_bonus = 0

        if query in candidate:
            substring_bonus += 25

        # 5. phonetic bonus
        phonetic_bonus = 0

        if (
            jellyfish.soundex(query)
            ==
            jellyfish.soundex(candidate)
        ):
            phonetic_bonus += 10

        # weighted score
        final_score = (
            ratio_score * 0.55
            + partial_score * 0.20
            + prefix_bonus
            + substring_bonus
            + phonetic_bonus
        )

        return round(
            min(final_score, 100),
            2
        )

    # -------------------------
    # Search
    # -------------------------
    def find_similar_and_save(
        self,
        input_email,
        output_csv="output.csv",
        min_score=55,
        top_n=50
    ):

        query = self._clean_email(
            input_email
        )

        results = []

        for _, row in self.df.iterrows():

            candidate = row["username"]

            score = self._score(
                query,
                candidate
            )

            if score >= min_score:

                results.append({
                    "panelistid":
                        row[self.panelist_col],

                    "email":
                        row[self.email_col],

                    "score":
                        score
                })

        result_df = pd.DataFrame(
            results
        ).sort_values(
            "score",
            ascending=False
        )

        result_df = result_df.head(
            top_n
        )

        result_df.to_csv(
            output_csv,
            index=False
        )

        print(
            f"Saved {len(result_df)} "
            f"matches to {output_csv}"
        )

        return result_df


# -------------------------
# Usage
# -------------------------

matcher = EmailMatcher(
    csv_path="/content/panelists.csv")

matcher.find_similar_and_save(
    "glenparez13@outlook.com",
    "matched.csv"
)

/tmp/ipykernel_1025/1937717981.py:18: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_csv(csv_path)


Saved 13 matches to matched.csv


,panelistid,email,score
1,156132,glenparez639@gmail.com,100.00
4,156112,glenparez09@gmail.com,100.00
3,160017,glenparez13@gmail.com,100.00
6,157711,glenparez428@gmail.com,100.00
5,160335,glenparez72@gmail.com,100.00
2,94297,glenpalmer305@gmail.com,81.53
12,145968,glenpowel46@gmail.com,75.95
0,23343,glenmprice@yahoo.co.uk,65.53
11,143836,glennabrego@gmail.com,64.06
8,67863,glenbredon@gmail.com,59.74


In [8]:
import pandas as pd
import re
from rapidfuzz import fuzz
import jellyfish


class EmailMatcher:

    def __init__(
        self,
        csv_path,
        email_col="email",
        panelist_col="panelistId"
    ):
        self.email_col = email_col
        self.panelist_col = panelist_col

        self.df = pd.read_csv(
        csv_path,
        low_memory=False)

        self.df["parsed"] = (
            self.df[email_col]
            .fillna("")
            .astype(str)
            .apply(self._parse_email)
        )

    # --------------------------
    # Parse Email
    # --------------------------
    def _parse_email(self, email):

        username = (
            str(email)
            .lower()
            .split("@")[0]
        )

        letters = "".join(
            re.findall(r"[a-z]+", username)
        )

        numbers = "".join(
            re.findall(r"\d+", username)
        )

        return {
            "raw": username,
            "letters": letters,
            "numbers": numbers
        }

    # --------------------------
    # Main Scoring
    # --------------------------
    def _score(
        self,
        query,
        candidate
    ):

        # exact match
        if (
            query["raw"]
            ==
            candidate["raw"]
        ):
            return 100.0

        q_letters = query["letters"]
        c_letters = candidate["letters"]

        # --------------------------------
        # EXISTING LOGIC (UNCHANGED)
        # --------------------------------
        ratio_score = fuzz.ratio(
            q_letters,
            c_letters
        )

        partial_score = fuzz.partial_ratio(
            q_letters,
            c_letters
        )

        phonetic_bonus = 0

        if (
            q_letters
            and c_letters
            and jellyfish.soundex(
                q_letters
            )
            ==
            jellyfish.soundex(
                c_letters
            )
        ):
            phonetic_bonus = 5

        letter_score = (
            ratio_score * 0.7
            +
            partial_score * 0.3
            +
            phonetic_bonus
        )

        letter_score = min(
            letter_score,
            100
        )

        # --------------------------------
        # NEW FALLBACK
        # handles:
        # bholedipak
        # dipakbole
        # --------------------------------
        token_sort_score = fuzz.token_sort_ratio(
            re.findall(r'.{3,}', q_letters),
            re.findall(r'.{3,}', c_letters)
        )

        # take best of both
        letter_score = max(
            letter_score,
            token_sort_score * 0.92
        )

        # --------------------------------
        # NUMBER SCORE
        # same logic
        # --------------------------------
        q_num = query["numbers"]
        c_num = candidate["numbers"]

        number_adjustment = 0

        if q_num and c_num:

            num_score = fuzz.ratio(
                q_num,
                c_num
            )

            if q_num == c_num:
                number_adjustment = 4

            elif num_score >= 70:
                number_adjustment = -2

            else:
                number_adjustment = -8

        final_score = (
            letter_score
            +
            number_adjustment
        )

        return round(
            max(
                min(final_score, 99.9),
                0
            ),
            2
        )


    # --------------------------
    # Search
    # --------------------------
    def find_similar_and_save(
        self,
        input_email,
        output_csv="matched.csv",
        min_score=55,
        top_n=50
    ):

        query = self._parse_email(
            input_email
        )

        results = []

        for _, row in self.df.iterrows():

            score = self._score(
                query,
                row["parsed"]
            )

            if score >= min_score:

                results.append({
                    "panelistid":
                        row[
                            self.panelist_col
                        ],

                    "email":
                        row[
                            self.email_col
                        ],

                    "score":
                        score
                })

    # -----------------------
    # HANDLE NO MATCH
    # -----------------------
        if len(results) == 0:

            print(
                "No match found"
            )

            empty_df = pd.DataFrame(
                columns=[
                    "panelistid",
                    "email",
                    "score"
                ]
            )

            empty_df.to_csv(
                output_csv,
                index=False
            )

            return empty_df

        result_df = pd.DataFrame(
            results
        )

        result_df = (
            result_df
            .sort_values(
                "score",
                ascending=False
            )
            .head(top_n)
        )

        result_df.to_csv(
            output_csv,
            index=False
        )

        print(
            f"Saved "
            f"{len(result_df)} "
            f"matches to "
            f"{output_csv}"
        )

        return result_df


# --------------------------
# Usage
# --------------------------

matcher = EmailMatcher(
    "/content/panelists.csv"
)

matcher.find_similar_and_save(
    "bholedipak@gmail.com"
)

Saved 50 matches to matched.csv


,panelistid,email,score
198,42991,pdipak1212@yahoo.com,71.02
38,93994,bledina01@hotmail.com,70.84
252,143868,hopediaz1820@gmail.com,69.17
25,55504,bholup533@gmail.com,67.75
233,134254,hollep777@gmail.com,65.57
158,18164,heidi.sparkes@hotmail.co.uk,65.55
56,14917,holmesdavidp@yahoo.com,64.55
210,86714,boyedmila7@gmail.com,64.21
251,134664,heidiakov@yahoo.com,64.21
228,55104,ramya.tholipaka@gmail.com,64.17
